In [1]:
#Install Required Libraries
# !pip install pdfplumber pymupdf pandas matplotlib
# !pip install spacy sentence-transformers
# !python -m spacy download en_core_web_sm

In [2]:
#Import
import re
import pdfplumber
import pandas as pd

from sentence_transformers import (
    SentenceTransformer,
    util
)

c:\Users\Prerna Thakur\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Load Sentence Transformer Model
model=SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3765.54it/s]


In [4]:
#Skills Database
ALL_SKILLS=[
    #IT
    "python", "java", "c++", "sql", "html", "css", "javascript", "react", "django", "flask",

    #AI
    "machine learning ", "deep learning", "nlp", "transorflow", "pytorch",

    #Accounting
    "accoumting", "gst", "taxation", "tally",  "bookkeeing",

    #HR
    "recriitment", "payroll", "employee engagement",

    #Marketing
    "seo", "digital marketing", "branding", "analytics",

    #Teaching
    "teaching", "lesson planning", "classroom management"
]

In [5]:
#Resume text extraction
def extract_text_from_pdf(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:

                text += page_text + "\n"

    return text           

In [6]:
#Name extraction
def extract_name(text):
    lines = text.split("\n")

    if lines:
        return lines[0]
    return "Not Found"

In [7]:
#Email extraction
def extract_email(text):

    match = re.search(
        r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]+",text)
    
    if match:
        return match.group()
    return"Not Found"

In [8]:
#Phone extarction
def extract_phone(text):

    match =re.search(
        r"\+?\d[\d -]{8,12}\d",
        text
    )
    if match:
        return match.group()
    return "Not Found"

In [18]:
#Skill Extraction
def extract_skills(text):
    text_lower = text.lower()
    found_skills=[]
    for skill in  ALL_SKILLS:
        if skill.lower() in text_lower:
            found_skills.append(skill)
    return list(set(found_skills))
        
        

In [19]:
#Education extraction
import re
import pandas as pd

def extract_education(text):

    education_keywords = [

        "10th",
        "12th",
        "bca",
        "mca",
        "btech",
        "mtech",
        "bsc",
        "msc",
        "ba",
        "ma",
        "bcom",
        "mcom",
        "mba",
        "phd",
        "pgdca",
        "diploma"
    ]

    lines = text.split("\n")

    education_data = []

    for line in lines:

        line_lower = line.lower()

        for edu in education_keywords:

            if edu in line_lower:

                # Find percentage
                percentage_match = re.search(
                    r"(\d{2,3}\.?\d*)\s?%",
                    line
                )

                percentage = "N/A"

                if percentage_match:

                    percentage = percentage_match.group(1)

                education_data.append({

                    "Qualification": edu.upper(),

                    "Percentage": percentage
                })

    # Create DataFrame
    df = pd.DataFrame(education_data)

    # Remove duplicates
    if not df.empty:

        df = df.drop_duplicates()

    return df

In [20]:
#Experience Extraction
def extract_experience(text):
    match = re.search(
        r"(\d+)\+?\s+years",
        text.lower()
    )
    if match :
        return int(match.group(1))
    return 0

In [21]:
#Skill Matching
def normalize_skills(skills):

    return list(set([

        str(skill).lower().strip()

        for skill in skills

        if str(skill).strip()
    ]))
def extract_skills(text):

    text_lower = text.lower()

    extracted_skills = []

    for skill in ALL_SKILLS:

        if skill.lower() in text_lower:

            extracted_skills.append(
                skill.lower()
            )

    return normalize_skills(
        extracted_skills
    )

def match_skills(

    resume_skills,
    required_skills
):

    resume_skills = normalize_skills(
        resume_skills
    )

    required_skills = normalize_skills(
        required_skills
    )

    matched = []

    for skill in resume_skills:

        if skill in required_skills:

            matched.append(skill)

    return list(set(matched))

In [22]:
#JD Similarity
def jd_similarity(jd_text, resume_text):
    jd_embedding = model.encode(jd_text, convert_to_tensor=True)
    resume_embedding = model.encode(resume_text, convert_to_tensor =True)
    similarity = util.cos_sim(jd_embedding, resume_embedding)

    return float(similarity[0][0])*100

In [23]:
#Final Scoring
def calculate_score(

    matched_skills,
    required_skills,
    education,
    experience,
    jd_score
):

    # =========================================
    # SKILL SCORE
    # =========================================

    if len(required_skills) > 0:

        skill_score = (
            len(matched_skills)
            / len(required_skills)
        ) * 100

    else:

        skill_score = 0

    # =========================================
    # EDUCATION SCORE
    # =========================================

    edu_score = 0

    if education is not None:

        try:

            if not education.empty:

                edu_score = 20

        except:

            edu_score = 0

    # =========================================
    # EXPERIENCE SCORE
    # =========================================

    exp_score = min(
        experience * 10,
        20
    )

    # =========================================
    # FINAL SCORE
    # =========================================

    final_score = (

        (skill_score * 0.5)

        + (jd_score * 0.3)

        + (edu_score * 0.1)

        + (exp_score * 0.1)
    )

    return (

        round(skill_score, 2),

        round(edu_score, 2),

        round(exp_score, 2),

        round(final_score, 2)
    )

In [24]:
#upload Resume
resume_path = r"D:\Prerna\Resume\IT\Black Modern Professional Resume.pdf"
jd_text = """
Python Developer with HTML and Java skills
"""

required_skills = normalize_skills([
    "Python",
    "HTML",
    "Java"
])

In [25]:
#Run Complete Testing
resume_text = extract_text_from_pdf(
    resume_path
)

name = extract_name(resume_text)

email = extract_email(resume_text)

phone = extract_phone(resume_text)

education = extract_education(resume_text)

experience = extract_experience(resume_text)

skills = extract_skills(resume_text)

matched_skills = match_skills(
    skills,
    required_skills
)

jd_score = jd_similarity(
    jd_text,
    resume_text
)

skill_score, edu_score, exp_score, final_score = calculate_score(

    matched_skills,
    required_skills,
    education,
    experience,
    jd_score
)

In [26]:
print("NAME:", name)

print("EMAIL:", email)

print("PHONE:", phone)

print("SKILLS:", skills)

print("\nEDUCATION:\n")

print(education)

print("\nEXPERIENCE:", experience)

print("\nMATCHED SKILLS:", matched_skills)

print("\nJD SCORE:", round(jd_score, 2))

print("\nFINAL SCORE:", final_score)

NAME: PRERNA
EMAIL: thakurprerna444@gmail.com
PHONE: +91-7650053193
SKILLS: ['html', 'python', 'java', 'c++']

EDUCATION:

  Qualification Percentage
0          10TH      81.14
1          12TH      90.04
2            MA        N/A

EXPERIENCE: 0

MATCHED SKILLS: ['html', 'python', 'java']

JD SCORE: 27.99

FINAL SCORE: 60.4
